# Churn Predictor
This notebook aims to study the data gathered from a Bank to predict customer churn.

The game plan is the following:
1. Exploratory Data Analysis (EDA)
2. **Data Preprocessing**
    + Data Unsampling using SMOTE
    + Principal Component Analysis Of One Hot Encoded Data
3. Model Selection and Evaluation
    + Cross Validation
    + Model Evaluation
    + Model Evaluation On Original Data (Before Upsampling)
4. Results

⚠️ Correct order
1. Train / Test split
2. Encoding (One-Hot if there are categorical ones)
3. SMOTE only in TRAIN
4. Scaling (if applicable)
5. PCA (optional, depending on the model)

In [30]:
import pandas as pd

data = pd.read_csv("BankChurners.csv")

# Target
target = "Attrition_Flag"

# Numerical features (EDA-driven)
numerical_features = [
    # Transaction behavior
    "Total_Trans_Ct",
    "Total_Trans_Amt",
    "Avg_Utilization_Ratio",
    "Total_Ct_Chng_Q4_Q1",
    "Total_Amt_Chng_Q4_Q1",
    "Total_Revolving_Bal",

    # Engagement & relationship
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Total_Relationship_Count",

    # Demographic / tenure
    "Customer_Age",
    "Months_on_book",
    "Dependent_count",
]

# Categorical features (EDA-selected)
categorical_features = [ # Marital_Status --> weak signaL
    "Gender",
    "Education_Level",
    "Income_Category",
    "Card_Category",
]


# Build modeling dataframe
df = data[numerical_features + categorical_features + [target]].copy()

print("Shape:", df.shape)
print("\nTarget distribution:")
print(df[target].value_counts(normalize=True))

df.head()


Shape: (10127, 17)

Target distribution:
Attrition_Flag
Existing Customer    0.83934
Attrited Customer    0.16066
Name: proportion, dtype: float64


,Total_Trans_Ct,Total_Trans_Amt,Avg_Utilization_Ratio,Total_Ct_Chng_Q4_Q1,Total_Amt_Chng_Q4_Q1,Total_Revolving_Bal,Months_Inactive_12_mon,Contacts_Count_12_mon,Total_Relationship_Count,Customer_Age,Months_on_book,Dependent_count,Gender,Education_Level,Income_Category,Card_Category,Attrition_Flag
0,42,1144,0.061,1.625,1.335,777,1,3,5,45,39,3,M,High School,$60K - $80K,Blue,Existing Customer
1,33,1291,0.105,3.714,1.541,864,1,2,6,49,44,5,F,Graduate,Less than $40K,Blue,Existing Customer
2,20,1887,0.000,2.333,2.594,0,1,0,4,51,36,3,M,Graduate,$80K - $120K,Blue,Existing Customer
3,20,1171,0.760,2.333,1.405,2517,4,1,3,40,34,4,F,High School,Less than $40K,Blue,Existing Customer
4,28,816,0.000,2.500,2.175,0,1,0,5,40,21,3,M,Uneducated,$60K - $80K,Blue,Existing Customer


## Data Preprocessing Order in the Kaggle Notebook (Explanation)

This section explains **the exact logical order followed in the Kaggle notebook**
[*“Bank Churn Data Exploration and Churn Prediction”*](https://www.kaggle.com/code/thomaskonstantin/bank-churn-data-exploration-and-churn-prediction/notebook) and contrasts it with
best practices regarding data leakage.


## Order Followed in the Kaggle Notebook

In the notebook, **data preprocessing is applied to the full dataset before the train/test split**.
The order is conceptually the following:

### 1. One-Hot Encoding
- Categorical variables are converted into numerical format using one-hot encoding.
- This transformation is applied to the **entire dataset**.
- Result: a high-dimensional feature space.

### 2. SMOTE (Synthetic Minority Oversampling Technique)
- SMOTE is applied **after encoding**.
- Synthetic samples are generated to balance the minority class.
- This is still done on the **full dataset**, not only on training data.

### 3. Scaling
- Features are scaled (typically using `StandardScaler`) to normalize magnitudes.
- Scaling is required because PCA is sensitive to feature scale.

### 4. PCA on One-Hot Encoded Data
- Principal Component Analysis is applied to reduce dimensionality.
- PCA is fitted on the **already SMOTEd and scaled full dataset**.
- A percentage of explained variance (e.g., 95%) is retained.

### 5. Train / Test Split
- Only after all preprocessing steps, the data is split into training and test sets.
- Models are trained and evaluated on this split.


## Why This Order Is Problematic

Although this approach works for experimentation, it introduces **data leakage**:

- SMOTE generates synthetic samples using information from data that later appears in the test set.
- Scaling and PCA are fitted using statistics (mean, variance, covariance) computed on the full dataset.
- As a result, the test set is no longer truly “unseen”.

This can lead to **optimistic performance estimates**.


## Correct Order (Best Practice)

For a production-ready workflow, the correct order should be:

1. **Train / Test Split**
2. **Encoding (fit on train, transform train and test)**
3. **SMOTE (only on training data)**
4. **Scaling (fit on training data)**
5. **PCA (fit on training data, optional)**

This ensures:
- No information leakage
- Fair model evaluation
- Reproducibility and robustness


In [28]:
# 1. Train / Test split

from sklearn.model_selection import train_test_split

X = df.drop(columns = target)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # ensures that the class ratio remains the same in 
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)
print("\nClass distribution (train):")
print(y_train.value_counts(normalize=True))
print("\nClass distribution (test):")
print(y_test.value_counts(normalize=True))


Train: (8101, 16) (8101,)
Test:  (2026, 16) (2026,)

Class distribution (train):
Attrition_Flag
Existing Customer    0.839279
Attrited Customer    0.160721
Name: proportion, dtype: float64

Class distribution (test):
Attrition_Flag
Existing Customer    0.839585
Attrited Customer    0.160415
Name: proportion, dtype: float64


## 1. Train / Test Split

The dataset is split into a training set (80%) and a test set (20%).
A **stratified split** is used to preserve the original class distribution 
of the target variable in both subsets.

### Interpretation

The class distribution is nearly identical in both the training and test sets,
confirming that stratification was applied correctly. This is particularly
important in churn prediction problems, where the minority class
(attrited customers) is underrepresented.

Maintaining consistent class proportions ensures that model evaluation on the
test set remains reliable and comparable to training performance, preventing
biased or overly optimistic results.


In [41]:
# 2. Encoding (fit on train, transform train and test)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identify categorical and numerical columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns
num_cols = X_train.select_dtypes(exclude=["object", "category"]).columns

print("Categorical Columns: ", list(cat_cols))
print("Numerical columns:", list(num_cols))

# OneHot Encoder
encoder = ColumnTransformer( # EXPLICAR
    transformers=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]

)

# Fit on training data only
X_train_enc = encoder.fit_transform(X_train)

# Transform test data using learned categories
X_test_enc = encoder.transform(X_test)

print("\nEncoded train shape:", X_train_enc.shape)
print("Encoded test shape: ", X_test_enc.shape)


Categorical Columns:  ['Gender', 'Education_Level', 'Income_Category', 'Card_Category']
Numerical columns: ['Total_Trans_Ct', 'Total_Trans_Amt', 'Avg_Utilization_Ratio', 'Total_Ct_Chng_Q4_Q1', 'Total_Amt_Chng_Q4_Q1', 'Total_Revolving_Bal', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Total_Relationship_Count', 'Customer_Age', 'Months_on_book', 'Dependent_count']

Encoded train shape: (8101, 31)
Encoded test shape:  (2026, 31)


## 2. Encoding (One-Hot Encoding)

Categorical variables are transformed into numerical features using
one-hot encoding. The encoder is **fitted exclusively on the training set**
and then applied to both the training and test sets to ensure a consistent
feature space and to prevent information leakage.

Numerical variables are passed through without modification at this stage.

The use of `handle_unknown="ignore"` allows the model to safely handle
categories that may appear in the test set but were not observed during
training.

--- 

The `ColumnTransformer` is used to apply different preprocessing steps to
different subsets of features within the same dataset.

```python
encoder = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)
```

* **`ColumnTransformer`** allows multiple transformations to be applied in
  parallel to different column groups, ensuring a clean and reproducible
  preprocessing pipeline.

* **`("onehot", OneHotEncoder(...), cat_cols)`**:

  * Applies one-hot encoding to categorical variables.
  * Each category is converted into a binary indicator feature.
  * `handle_unknown="ignore"` ensures that unseen categories appearing in the
    test set do not cause errors and are encoded as all-zero vectors.

* **`("num", "passthrough", num_cols)`**:

  * Numerical features are passed through unchanged at this stage.
  * Scaling is intentionally deferred to a later step to avoid data leakage
    and to ensure compatibility with subsequent preprocessing (e.g., SMOTE and PCA).


In [47]:
# 3. SMOTE (only on training data)
from imblearn.over_sampling import SMOTE

# Check class distribution before SMOTE
print("Before SMOTE: ")
print(pd.Series(y_train).value_counts())

# Apply SMOTE only on the encoded training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_enc, y_train)

# Check class distribution before SMOTE
print("\nAfter SMOTE: ")
print(pd.Series(y_train_smote).value_counts())

Before SMOTE: 
Attrition_Flag
Existing Customer    6799
Attrited Customer    1302
Name: count, dtype: int64

After SMOTE: 
Attrition_Flag
Attrited Customer    6799
Existing Customer    6799
Name: count, dtype: int64


## 3. SMOTE (Synthetic Minority Oversampling Technique)

The target variable in this dataset is imbalanced, with a significantly
smaller proportion of attrited customers. To address this issue, SMOTE
(Synthetic Minority Oversampling Technique) is applied only to the training
data.

SMOTE works by generating synthetic samples of the minority class through
interpolation between existing minority observations in feature space. This
approach increases class balance without simply duplicating existing samples.

Importantly, SMOTE is applied **after one-hot encoding** (since it requires
numerical features) and **exclusively on the training set** to prevent data
leakage. The test set remains untouched and reflects the original class
distribution, ensuring a realistic and unbiased evaluation of model
performance.

After applying SMOTE, the training set contains a balanced number of samples
for each class, allowing the model to better learn patterns associated with
customer attrition.


In [48]:
# 4. Scaling (fit on training data)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit scaler on SMOTEd training data
X_train_scaled = scaler.fit_transform(X_train_smote)

# Apply the same transformation to test data
X_test_scaled = scaler.fit_transform(X_test_enc)

print("Scaled Train data shape: ", X_train_scaled.shape)
print("Scaled Test data shape: ", X_test_scaled.shape)

Scaled Train data shape:  (13598, 31)
Scaled Test data shape:  (2026, 31)


## 4. Feature Scaling

Feature scaling is applied to standardize the numerical range of all features.
This step is particularly important because the dataset contains a mix of
one-hot encoded binary variables and continuous numerical features.

The `StandardScaler` is used to transform features to have zero mean and unit
variance. The scaler is **fitted exclusively on the training data after SMOTE**
and then applied to both the training and test sets using the same learned
parameters.

Fitting the scaler after SMOTE ensures that synthetic samples generated during
oversampling are scaled consistently with real training observations. Applying
the same transformation to the test set preserves comparability while
preventing information leakage.

Scaling is a necessary preprocessing step for algorithms that rely on distance
or variance structure, such as PCA and linear models.


In [49]:
# 5. PCA (fit on training data, optional)
from sklearn.decomposition import PCA 

# Initialize PCA to retain 95% of the variance
pca = PCA(n_components=0.95, random_state=42)


# Fit PCA on scaled training data
X_train_pca = pca.fit_transform(X_train_scaled)

# Fit PCA on scaled test data
X_test_pca = pca.transform(X_test_scaled)


print("Train shape after PCA:", X_train_pca.shape)
print("Test shape after PCA: ", X_test_pca.shape)
print("Explained variance ratio:", pca.explained_variance_ratio_.sum())


Train shape after PCA: (13598, 22)
Test shape after PCA:  (2026, 22)
Explained variance ratio: 0.9564072871723206


## 5. Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is an optional dimensionality reduction
technique applied after feature scaling. PCA projects the original feature
space into a lower-dimensional set of orthogonal components that capture the
maximum variance in the data.

### Why PCA Is Optional

PCA is not strictly required for all machine learning models. Tree-based models
(e.g., Random Forest, Gradient Boosting) are generally robust to high
dimensionality and correlated features. However, linear models and
distance-based methods can be negatively affected by multicollinearity and a
large number of input variables.

Therefore, PCA is considered optional and model-dependent.

### Why PCA Is Applied in This Project

In this dataset, one-hot encoding of categorical variables results in a
high-dimensional feature space. Applying PCA provides several benefits:

- Reduces dimensionality and model complexity
- Mitigates multicollinearity introduced by one-hot encoding
- Improves numerical stability for linear models
- Potentially reduces overfitting and training time

PCA is fitted **only on the scaled training data** to avoid information leakage,
and the same transformation is applied to the test set. The number of
components is chosen to retain 95% of the total variance, ensuring that most of
the informative structure of the data is preserved while reducing noise.

As a result, PCA provides a more compact and efficient representation of the
data, which is particularly suitable for subsequent modeling and evaluation.
